## Azure ML - Training in PyTorch
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we demonstrate training a pytorch model in AzureML. This is a variation on the Climate Zones tutorial found elsewhere in this repository.

### Environment 
This notebook uses the environment defined in this repository in the [environments/requirements_pytorch.yaml](../../../environments/requirements_pytorch.yaml) conda file, togther with the AzureML SDK. Use the `azure_conda_setup.sh` script to create a local environment on a compute instance.

### Imports

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import numpy 
import pandas

In [3]:
import matplotlib
import matplotlib.pyplot

In [4]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [5]:
import mlflow

/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import torch

## COnnect to AzureML workspace
We are going to use various features of AzureML, especially for data access and experiment tracking. 

In [8]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential


In [9]:

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json


## Load and prepare data 
We will now load the dataset and do the usual data prep steps, like train/test split and normalisation.


#### Dataset parameters

In [17]:
with open(pathlib.Path.cwd().parent / 'climate_zones' / 'config.json') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25a/mmh_storage/ai4c_data/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summ

In [18]:
data_asset = ml_client.data.get("climate_zones_1_0", version="1")

In [19]:
data_asset.path

'azureml://subscriptions/9734ed68-621d-47ed-babd-269110dbacb1/resourcegroups/1-da6c1ccf-playground-sandbox/workspaces/dscoptest1/datastores/dscopworkspacestore/paths/climate_zones/climate_zones_1p0.csv'

In [20]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


#### Load data for training

In [21]:
current_res = 1.0

In [22]:
mlready_data_path = df = data_asset.path

In [23]:
zones_df = pandas.read_csv(mlready_data_path)

Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

#### Selecting features

In [24]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [25]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [26]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

#### Train/test split

In [27]:
random_seed = tutorial_config['random_seed']

In [28]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [29]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [30]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


## Using PyTorch

From this point in the tutorial, we diverge from what was done in the ml training pipeline tutorial as instead of setting up and training our model in scikit-learn, we're going to use a more sophisticated machine learning library called pytorch. This gives us more more control over how we implement our neural network and gives us much more power to use advanced architectures, losss functions and distributed computing techniques.

When using PyTorch, the main difference is that we will specify the details of the neural network architecture and training loop much more explictly. This means we can customise and optimise these details for the particular problem. Key elements of a PyTorch training pipeline, compared to what was previous described are as follows:
1. **Data Loading and Cleaning** - This is usually done through a PyTorch Dataset class. In addition, you create a Dataset Loader object, which has the responsibility for iterating through the data in the dataset, including shuffling data between epochs where appropriate.</p>
2. **Feature Engineering** - Same as before, but may be a part of the dataset class.</p>
3. **Train/Test split** - Same method as before. Usually different dataset objects will represent the train, validate and test sets.</p>
4. **Data Preparation** - Same as before, but code may be structured differently such that the normalisation and scaling happening inside the PyTorch dataset class.</p>
5. **Algorithm Setup** - Usually you create a class representing the model architecture. You then also specify key hyperparameters such as the optimiser for training the weights, the learning rate, etc. </p>
6. **Algorithm Training** - The elements of the training loop are described more explictly in PyTorch typically with a explicit loop for iterating through batches and an outer loop for iterating through batches.</p>
7. **Inference** - The model object is used as a callable object for producing. </p>
8. **Evaluation** - Evaluation is the same as before. </p>
9. **Interpretability and Explainability** - As model architectures become more complex, it becomes more difficult to explain or interpret the results, this is an active area of research. </p>
10. **Model Storage** - Pytorch has a more sophisticated mechanism for [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html). </p>


### Check GPU availability

As a part of this notebook, we will now be training our models using the GPUs on JASMIN. There are some additional steps required for this purpose, such as moving the data and model onto the GPU memory for processing. ML frameworks are especially helpful for this in abtracting away many of the details of this into a few commands.

In this cell, we check whether there is a GPU to use. [CUDA](https://en.wikipedia.org/wiki/CUDA) is the underlying software layer that interfaces to nvidia GPUs. This check allows the notebook to seamlessly work either on a gpu if one is available or to do processing on a cpu when a gpu is not available.



In [31]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

#### Training hyperparameters

In [32]:
training_params = {
    'batch_size': 16,
    'num_epochs': 2,
    'learning_rate': 0.001,
    'loss': 'CrossEntropyLoss',
    'criterion': 'CrossEntropyLoss',
    'optimizer': 'Adam',
}

### Define a data loader

Pytorch defines the interface between the dataset and machine learning through a base class (`torch.utils.data.Dataset), which follow [pythonic paradigms](https://realpython.com/ref/glossary/pythonic/) for software architecture. The implementation of the class hides away the details of the data being used, so that the dataset can be used in the generic pytorch architecture. There is also then a data loader which is an [iterator](https://www.w3schools.com/python/python_iterators.asp) on the dataset, enabling pytorch to progress through all the data during training. Ultimately the aim of this data architecture is to present the data in the correct [pytorch tensor format](https://docs.pytorch.org/docs/stable/tensors.html) expected by the neural network for training purposes.

#### Key Concepts:
- **Pytorch Dataset**: Handles loading and preparing the data for use with pytorch. Implements key [python built-in methods](), including:</p>
  - [`__init__`](https://docs.python.org/3/reference/datamodel.html#object.__init__) creates the dataset object, and typically loads and transforms the data ready for use, or in a lazy loading paradigm, define the task pipeline for loading the data upon request.
  - [`__len__`](https://docs.python.org/3/reference/datamodel.html#object.__len__) specifies how many data point there are.
  - [`__getitem__`](https://docs.python.org/3/reference/datamodel.html#object.__getitem__) return the item for a particular index.</p>
- [**Pytorch Data Loader**](https://docs.pytorch.org/docs/stable/data.html): Interfaces between the ML algorithm being trained and the dataset, selecting mini batches of data to use with each iteration of the gradient descent with back propogation used for training the neural network. Key parameters include:</p>
  - **dataset** - The dataset to load the data from (as described above).
  - [**batch size**](https://www.geeksforgeeks.org/deep-learning/batch-size-in-neural-network/) - How many samples to use in each mini batch during training.
  - [**shuffle**](https://stats.stackexchange.com/questions/245502/why-should-we-shuffle-data-while-training-a-neural-network) - Whether to shuffle the order of data use during each of the epochs. Data shuffling improves the generalisation of what is learned by the neural network.


Further reading
- [Intro to datasets and data loader - pytorch docs](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html)
- [Tutorial on using CSV data with pytorch data architecture - Machine Learning Mastery](https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/)
- [Converting pnadas dataframe to the pytorch](https://www.geeksforgeeks.org/deep-learning/converting-a-pandas-dataframe-to-a-pytorch-tensor/)
- [Encoding target data for a neural network using LabelBinarizer - scikit learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html)

### Implementation details for this tutorial

In [33]:
class ClimateZonesDataset(torch.utils.data.Dataset):
    """
    Inspired by this tutorial:
    https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
    """
    def __init__(self, df_ml, predictor_features, target_feature, device, stats_dict=None):
        self._df_ml = df_ml.reset_index().drop(['index'],axis='columns')
        self._device = device
        self.input_scaler = sklearn.preprocessing.StandardScaler()
        if stats_dict is None:
            self.input_scaler.fit(self._df_ml[predictor_features])
        else:
            self.input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
            self.input_scaler.scale_ = numpy.array(stats_dict['input_scale'])

        self._X = torch.tensor(self.input_scaler.transform(self._df_ml[predictor_features]),  
                               dtype=torch.float32)


        self.target_encoder = sklearn.preprocessing.LabelBinarizer(sparse_output=False)
        if stats_dict is None:
            self.target_encoder.fit(self._df_ml[[target_feature]])
        else:
            self.target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

        
        self._y = torch.tensor(self.target_encoder.transform(self._df_ml[[target_feature]]),
                               dtype=torch.float32)

        self.stats_dict = {
            'input_mean': [float(v1) for v1 in self.input_scaler.mean_],
            'input_scale': [float(v1) for v1 in self.input_scaler.scale_],
            'target_classes': list(self.target_encoder.classes_),
        }
        
    def _repr_html_(self):
        return f'''
        <h1>Climate Zones Dataset</h1>
        Number of samples {len(self._X)}
        '''
    
    def __len__(self):
        return len(self._X)

    def __getitem__(self,idx):
        return self._X[idx], self._y[idx]


        

We now intialise the validate and test set data loaders. Note that we initialise the preprocessing objects with the values learned from the training data, rather than calculating them on the validate or test data.

### Using our dataset class
Once we have defined the class, we can now initialise objects from it. A typical pattern is to define separate objects for the train, validate and test sets. You then define an iterator, i.e. a data loader object, for each of the dataset objects. Data loaders are a generic class defined by PyTorch that should work with any dataset that complies with the standard interface.

In [34]:
cz_train_ds = ClimateZonesDataset(train_df, predictors, target_var, device)
cz_train_ds

In [35]:
cz_val_ds = ClimateZonesDataset(val_df, predictors, target_var, device, cz_train_ds.stats_dict)
cz_test_ds = ClimateZonesDataset(test_df, predictors, target_var, device, cz_train_ds.stats_dict)

/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [36]:
len(cz_train_ds)

325206

In [37]:
cz_val_ds[1234]

(tensor([-0.5431, -0.5180, -0.4127, -0.3780, -0.4759, -0.5458, -0.5103, -0.4938,
         -0.5388, -0.5015, -0.5389, -0.5469, -0.5396, -0.7533, -0.9728, -1.0988,
         -1.2024, -1.2934, -1.3218, -1.3201, -1.2447, -1.1500, -0.9019, -0.6197]),
 tensor([0., 0., 0., 0., 1.]))

In [38]:
cz_val_ds[1234][0]

tensor([-0.5431, -0.5180, -0.4127, -0.3780, -0.4759, -0.5458, -0.5103, -0.4938,
        -0.5388, -0.5015, -0.5389, -0.5469, -0.5396, -0.7533, -0.9728, -1.0988,
        -1.2024, -1.2934, -1.3218, -1.3201, -1.2447, -1.1500, -0.9019, -0.6197])

In [39]:
cz_val_ds[1234][0].shape

torch.Size([24])

In [40]:
cz_val_ds[1234][1]

tensor([0., 0., 0., 0., 1.])

In [41]:
cz_val_ds[1234][1].shape

torch.Size([5])

In [42]:
cz_train_loader = torch.utils.data.DataLoader(
        cz_train_ds, batch_size=training_params['batch_size'], shuffle=True, num_workers=1,
    )
cz_val_loader = torch.utils.data.DataLoader(
        cz_val_ds, batch_size=training_params['batch_size'], shuffle=False, num_workers=1,
    )

In [43]:
training_params['batch_size']

16

In [44]:
count = 0
for X1,y1 in cz_train_loader:
    print(X1, y1)
    count +=1
    if count > 5:
        break

Exception ignored in: <function PeriodicExportingMetricReader.__init__.<locals>.<lambda> at 0x740c27d22d40>
Traceback (most recent call last):
  File "/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/opentelemetry/sdk/metrics/_internal/export/__init__.py", line 535, in <lambda>
    after_in_child=lambda: weak_at_fork()()  # pylint: disable=unnecessary-lambda
                           ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable


tensor([[-0.1645, -0.2294, -0.2181, -0.0487,  0.0351,  0.3103,  0.1461,  0.2793,
          0.1426,  0.1837,  0.0162, -0.1511, -0.6712, -0.4738, -0.0609,  0.1014,
          0.2748,  0.4694,  0.5495,  0.4394,  0.3160,  0.0970, -0.3120, -0.6072],
        [-0.5727, -0.5729, -0.5796, -0.5782, -0.6083, -0.6142, -0.6318, -0.6705,
         -0.6582, -0.6390, -0.6281, -0.6078, -0.2396, -0.5042, -0.7767, -0.9431,
         -1.0454, -1.1322, -1.1814, -1.1710, -1.0908, -0.9519, -0.6417, -0.3217],
        [ 0.0628, -0.1159, -0.1524, -0.0827,  0.0985,  0.2200,  0.1475,  0.1763,
          0.1779,  0.3766,  0.2506,  0.1396, -0.1845, -0.0642,  0.1401,  0.3075,
          0.4451,  0.5296,  0.5825,  0.4932,  0.3962,  0.2583,  0.0610, -0.0959],
        [-0.1408, -0.0768,  0.1052,  0.1659,  0.0967, -0.1764, -0.1756, -0.1530,
         -0.0501, -0.0171, -0.1084, -0.1237, -0.1723, -0.3576, -0.5855, -0.7598,
         -0.8928, -1.0032, -1.0471, -1.0323, -0.9369, -0.8126, -0.5520, -0.2527],
        [ 0.2538,  0.310

## Building a pytorch model
The next step is build a class to encapsulate the architecture of the ML model that we are going to train. As with the dataset, we do this by


### Further reading
- [Classification tutorial - Pytorch docs](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [Classification tutorial - Machine Learning Mastery](https://machinelearningmastery.com/building-a-multiclass-classification-model-in-pytorch/ )

In [45]:
class ClimateZoneClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(24, 60)
        self.act1 = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(60, 60)
        self.act2 = torch.nn.ReLU()
        self.layer3 = torch.nn.Linear(60, 60)
        self.act3 = torch.nn.ReLU()
        self.output = torch.nn.Linear(60, 5)
        self.sigmoid = torch.nn.Sigmoid()
        self.lsm = torch.nn.Softmax(dim=-1)
 
    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        # x = self.sigmoid(self.output(x))
        x = self.lsm(self.output(x))
        return x
    


In [46]:
cz_classifier = ClimateZoneClassifier().to(device)

In [47]:
cz_classifier

ClimateZoneClassifier(
  (layer1): Linear(in_features=24, out_features=60, bias=True)
  (act1): ReLU()
  (layer2): Linear(in_features=60, out_features=60, bias=True)
  (act2): ReLU()
  (layer3): Linear(in_features=60, out_features=60, bias=True)
  (act3): ReLU()
  (output): Linear(in_features=60, out_features=5, bias=True)
  (sigmoid): Sigmoid()
  (lsm): Softmax(dim=-1)
)

In [48]:
cz_classifier(cz_train_ds[:10][0].to(device))

tensor([[0.2137, 0.1695, 0.2110, 0.1922, 0.2135],
        [0.2135, 0.1696, 0.2111, 0.1921, 0.2136],
        [0.2134, 0.1695, 0.2114, 0.1920, 0.2138],
        [0.2132, 0.1695, 0.2115, 0.1919, 0.2139],
        [0.2131, 0.1694, 0.2118, 0.1918, 0.2139],
        [0.2127, 0.1695, 0.2119, 0.1917, 0.2142],
        [0.2125, 0.1694, 0.2120, 0.1915, 0.2145],
        [0.2122, 0.1695, 0.2120, 0.1913, 0.2149],
        [0.2122, 0.1695, 0.2121, 0.1913, 0.2150],
        [0.2120, 0.1695, 0.2121, 0.1911, 0.2152]], grad_fn=<SoftmaxBackward0>)

In [49]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cz_classifier.parameters(), 
                             lr=training_params['learning_rate'])

### Experiment tracking with mlflow
In a project to develop a machine learning model, we are likely to train many different models as part of our experimentation, with different predictors, hyperparameters, architectures and other experimental choices that we vary to understand the problem and find the best solution. We will have a collection of different models, and to properly assess our experiments we need to know exactly which set of choices go with which model. Experiment tracking tools log all the elements of a training run together so they can be retrieved and analysed later. A common tool for this is **ML Flow**.

The way this typically work is as follows:
- Create an experiment
- For each training run, start a run within the experiment
    - log the hyperparameter for the training run e.g. number of epochs, batch size, learning rate etc.
    - log loss and other metrics during training (every epoch typically to show loss descreases with training).
    - log final metrics at the end of training
    - create plots and add to the run
    - log the model with the run
- After multiple runs, you can search through the experiment to find the best run, according to a metric of your choice.
- You can select models from some runs to register for subsequent use, as a way of "releasing" the model.
- You can load model from runs, or registered models, and use them for inference.

Behind the scenes, ML flow has two key components. 
- **database** - this stores this info about runs and experiments
- **artifact store** - This save all disk objects, such as model weights or plots, that don't get stored in the database. Each artifact path is stored in the database to associate it with the experiment.

Further Reading
- [ML Flow docs](https://mlflow.org/)
- [Tracking pytorch with mlflow](https://mlflow.org/docs/latest/ml/deep-learning/pytorch/)


### ML Flow server

In this tutorial, we're going to use the MLFlow server built into AzureML. We can then browse through our experiments from the **Jobs** functionality in the AzureML Studio GUI.

In [10]:
mlflow_server_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri

In [12]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)

connecting to mlflow server azureml://southcentralus.api.azureml.ms/mlflow/v2.0/subscriptions/9734ed68-621d-47ed-babd-269110dbacb1/resourceGroups/1-da6c1ccf-playground-sandbox/providers/Microsoft.MachineLearningServices/workspaces/dscoptest1


In [13]:
mlflow.pytorch.autolog()

In [14]:
exp_name='climate_zones_torch_nn'

In [15]:
if mlflow.get_experiment_by_name(exp_name) is None:
    exp_id = mlflow.create_experiment(exp_name)
exp1 = mlflow.get_experiment_by_name(exp_name)
exp1

<Experiment: artifact_location='', creation_time=1787691982330, experiment_id='c7fb9cfc-27ad-4472-aaf7-51d8121a6f11', last_update_time=None, lifecycle_stage='active', name='climate_zones_torch_nn', tags={}>

In [50]:
cz_signature = mlflow.models.infer_signature(cz_train_ds[:5][0].numpy(), cz_train_ds[:5][1].numpy())

## Run the training loop

In PyTorch, we will explicitly define a loop which cycles through the data and updates the weights of the network to produce better predictions, which is to say predictions that are closer to the target data provided. Key steps in the training loop are:

- Iterate through the full data set for the number of epochs specified. An epoch is one full pass through the data.
- In each epoch, divide the dataset into minibatches. This is a set of data on which gradient descent is done.
- For each minibatch, do the follwing steps:
  -  Do a forward pass, or inference step to make predictions with the current weights. At the start of the process these will essentially be nonsense, but as we refine the weights the performance improves.
  -  Calculate the loss or error of the predictions.
  -  Do back propogation to update the weights towards weights that would produce correct predictions. This is done by calculating the gradient in the weights and moving the weights. The amount one updates the weight is based on the learning rate.
  -  Update the weights based on the calculated gradients.

### Further reading
- [Training a model - pytorch docs](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)
- [ Introduction to NNs - Kaggle](https://www.kaggle.com/code/carlosaguayo/introduction-to-neural-networks/notebook)
- [Tutorial on NNs for weather](https://github.com/MetOffice/ml_weather_tutorial/blob/main/03_algorithm_selection.ipynb)

In [51]:
def get_classification_metrics(model, cz_data, set_label, target_encoder):
    class_labels = target_encoder.classes_
    return pandas.DataFrame({
        'climate_group': class_labels,
        f'precision_{set_label}': sklearn.metrics.precision_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        f'recall_{set_label}': sklearn.metrics.recall_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
    })

In [52]:
num_epochs = training_params['num_epochs']
# num_epochs = 1 # for debug

In [53]:
%%time
with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
    print(current_run.info.run_id)
    mlflow.log_params(training_params)    
    mlflow.log_dict(cz_train_ds.stats_dict, 'stats.json')
    for epoch in range(num_epochs):
        print(f'epoch {epoch}')
        cz_classifier.train()
        epoch_loss_train = 0.0
        for batch_X, batch_y in cz_train_loader:
            optimizer.zero_grad()
            predictions = cz_classifier(batch_X.to(device))
            loss = loss_fn(predictions, batch_y.to(device))
            loss.backward()
            optimizer.step()
            epoch_loss_train += loss.to('cpu').item()

        #divide by number of batches
        epoch_loss_train /= len(cz_train_loader)
        
        epoch_loss_val = 0.0
        for X_val, y_val in cz_val_loader:
            epoch_loss_val += loss_fn(cz_classifier(X_val.to(device)), y_val.to(device)).item()    
        epoch_loss_val /= len(cz_val_loader)

        mlflow.log_metrics(
            { 'cross_entropy_train': epoch_loss_train,
            'cross_entropy_val': epoch_loss_val, },
            step=epoch,
        )
            
    metrics_df = get_classification_metrics(cz_classifier,
                                               cz_train_ds, 
                                               'train', 
                                               target_encoder=cz_train_ds.target_encoder)
    
    metrics_df = metrics_df.merge( get_classification_metrics(cz_classifier,
                                               cz_val_ds, 
                                               'val', 
                                               target_encoder=cz_train_ds.target_encoder), on='climate_group')
    mlflow.log_table(metrics_df, 'metrics.csv')
        
    torch.save(cz_classifier,'cz_model.pth')
    mlflow.log_artifact('cz_model.pth')
    
    # mlflow.pytorch.log_model(cz_classifier,
    #                      name='climate_zones_classifier_torch', 
    #                      # signature=cz_signature, 
    #                      # input_example = cz_train_ds[:5][0].numpy(),
    #                      # export_model=True,
    #                     )


573b9d54-7191-40ec-94d6-8ecabe34a24a
epoch 0


Exception ignored in: <function PeriodicExportingMetricReader.__init__.<locals>.<lambda> at 0x740c27d22d40>
 Traceback (most recent call last):
  File "/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/opentelemetry/sdk/metrics/_internal/export/__init__.py", line 535, in <lambda>
    after_in_child=lambda: weak_at_fork()()  # pylint: disable=unnecessary-lambda
                          ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable
Exception ignored in: <function PeriodicExportingMetricReader.__init__.<locals>.<lambda> at 0x740c27d22d40>
Traceback (most recent call last):
  File "/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/opentelemetry/sdk/metrics/_internal/export/__init__.py", line 535, in <lambda>
    after_in_child=lambda: weak_at_fork()()  # pylint: disable=unnecessary-lambda
                           ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable


epoch 1


Exception ignored in: <function PeriodicExportingMetricReader.__init__.<locals>.<lambda> at 0x740c27d22d40>
 Traceback (most recent call last):
  File "/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/opentelemetry/sdk/metrics/_internal/export/__init__.py", line 535, in <lambda>
    after_in_child=lambda: weak_at_fork()()  # pylint: disable=unnecessary-lambda
                          ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable
Exception ignored in: <function PeriodicExportingMetricReader.__init__.<locals>.<lambda> at 0x740c27d22d40>
Traceback (most recent call last):
  File "/anaconda/envs/dscop_pytorch/lib/python3.12/site-packages/opentelemetry/sdk/metrics/_internal/export/__init__.py", line 535, in <lambda>
    after_in_child=lambda: weak_at_fork()()  # pylint: disable=unnecessary-lambda
                           ^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not callable


🏃 View run gray_mango_85c2h0fy at: https://southcentralus.api.azureml.ms/mlflow/v2.0/subscriptions/9734ed68-621d-47ed-babd-269110dbacb1/resourceGroups/1-da6c1ccf-playground-sandbox/providers/Microsoft.MachineLearningServices/workspaces/dscoptest1/#/experiments/c7fb9cfc-27ad-4472-aaf7-51d8121a6f11/runs/573b9d54-7191-40ec-94d6-8ecabe34a24a
🧪 View experiment at: https://southcentralus.api.azureml.ms/mlflow/v2.0/subscriptions/9734ed68-621d-47ed-babd-269110dbacb1/resourceGroups/1-da6c1ccf-playground-sandbox/providers/Microsoft.MachineLearningServices/workspaces/dscoptest1/#/experiments/c7fb9cfc-27ad-4472-aaf7-51d8121a6f11
CPU times: user 6min 55s, sys: 41.9 s, total: 7min 36s
Wall time: 2min 46s


In [54]:
torch.save(cz_classifier,'cz_model.pth')
mlflow.log_artifact('cz_model.pth')


### Load model and do inference

After the run, we can look at the experiment to see details of the run. On some systems we can use a GUI to inspect results, by going to the same URI at the server, but on many systems the GUI is not available, so we can access the results through the python interface. Here we search through runs in an experiment, and look up a metric for the recent run.

In [55]:
mlflow.search_runs(exp1.experiment_id)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.cross_entropy_train,metrics.cross_entropy_val,params.criterion,params.optimizer,params.learning_rate,params.loss,params.num_epochs,params.batch_size,tags.mlflow.rootRunId,tags.mlflow.loggedArtifacts,tags.mlflow.runName,tags.mlflow.user
0,573b9d54-7191-40ec-94d6-8ecabe34a24a,c7fb9cfc-27ad-4472-aaf7-51d8121a6f11,FINISHED,,2026-08-25 21:09:26.143000+00:00,2026-08-25 21:12:12.500000+00:00,0.941622,0.951538,CrossEntropyLoss,Adam,0.001,CrossEntropyLoss,2,16,573b9d54-7191-40ec-94d6-8ecabe34a24a,"[{""path"": ""metrics.json"", ""type"": ""table""}]",gray_mango_85c2h0fy,Cloud Student c3fd45b1


In [56]:
model_inference = torch.load('cz_model.pth', weights_only=False)


In [57]:
model_inference

ClimateZoneClassifier(
  (layer1): Linear(in_features=24, out_features=60, bias=True)
  (act1): ReLU()
  (layer2): Linear(in_features=60, out_features=60, bias=True)
  (act2): ReLU()
  (layer3): Linear(in_features=60, out_features=60, bias=True)
  (act3): ReLU()
  (output): Linear(in_features=60, out_features=5, bias=True)
  (sigmoid): Sigmoid()
  (lsm): Softmax(dim=-1)
)

In [58]:
cz_train_ds.target_encoder.inverse_transform(
    model_inference(cz_val_ds._X.to(device)).to('cpu').detach().numpy()
)
        

array(['A', 'B', 'E', ..., 'E', 'C', 'D'], shape=(40644,), dtype='<U1')

### Evaluation

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

In [ ]:
# load metrics from mlflow to see training

In [59]:
current_run.info.run_id

'573b9d54-7191-40ec-94d6-8ecabe34a24a'

In [60]:
metrics_path = pathlib.Path(mlflow.artifacts.download_artifacts(f'runs:/{current_run.info.run_id}/metrics.json'))
metrics_path

PosixPath('/tmp/tmpu3dawl33/metrics.json')

In [61]:
with open(metrics_path) as metrics_file:
    metrics_mlflow_dict = json.load(metrics_file)
metrics_mlflow_dict    

{'columns': ['climate_group',
  'precision_train',
  'recall_train',
  'precision_val',
  'recall_val'],
 'data': [['A', 0.9466113485, 0.9473644953, 0.9470368183, 0.9429523623],
  ['B', 0.9436749465, 0.9737244119, 0.9443430657, 0.9733978235],
  ['C', 0.8587626596, 0.8432905737, 0.847275975, 0.8417627921],
  ['D', 0.9441472897, 0.9640816223, 0.9437305296, 0.9613248711],
  ['E', 0.991855079, 0.9662484064, 0.9910952181, 0.9658353263]]}

In [62]:
pandas.DataFrame(metrics_mlflow_dict['data'],columns=metrics_mlflow_dict['columns'])

,climate_group,precision_train,recall_train,precision_val,recall_val
0,A,0.946611,0.947364,0.947037,0.942952
1,B,0.943675,0.973724,0.944343,0.973398
2,C,0.858763,0.843291,0.847276,0.841763
3,D,0.944147,0.964082,0.943731,0.961325
4,E,0.991855,0.966248,0.991095,0.965835


In [67]:
mlflow.search_runs(exp1.experiment_id,
                   max_results=5,
                  )

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.cross_entropy_train,metrics.cross_entropy_val,params.criterion,params.optimizer,params.learning_rate,params.loss,params.num_epochs,params.batch_size,tags.mlflow.rootRunId,tags.mlflow.loggedArtifacts,tags.mlflow.runName,tags.mlflow.user
0,573b9d54-7191-40ec-94d6-8ecabe34a24a,c7fb9cfc-27ad-4472-aaf7-51d8121a6f11,FINISHED,,2026-08-25 21:09:26.143000+00:00,2026-08-25 21:12:12.500000+00:00,0.941622,0.951538,CrossEntropyLoss,Adam,0.001,CrossEntropyLoss,2,16,573b9d54-7191-40ec-94d6-8ecabe34a24a,"[{""path"": ""metrics.json"", ""type"": ""table""}]",gray_mango_85c2h0fy,Cloud Student c3fd45b1


In [ ]:
# calculate metrics from predictions on test vset

In [69]:
test_metrics_df = get_classification_metrics(model_inference,
                                             cz_test_ds, 
                                             'test', 
                                             target_encoder=cz_train_ds.target_encoder)
test_metrics_df


,climate_group,precision_test,recall_test
0,A,0.945941,0.949702
1,B,0.944143,0.975030
2,C,0.855220,0.841873
3,D,0.944782,0.961749
4,E,0.991991,0.966046


# Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Varying model architecture
Try to change the model architecture e.g. add more layers

In [ ]:
# insert code here

### Next steps or potential follow on material

Links from notebook
- [PyTorch docs](https://pytorch.org/)

Additional excercises in this tutorial material includes:
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)
